# Seismic Analysis Pipeline: SOM + Vision LLM + RAG

**Author:** [Your Name]  
**Dataset:** [OpenFWI — Kaggle Waveform Inversion Competition](https://www.kaggle.com/competitions/waveform-inversion)  
**Last updated:** 2025

---

## Overview

This notebook implements a **3-layer universal seismic analysis pipeline** applied to the OpenFWI synthetic dataset:

| Layer | Component | Purpose |
|-------|-----------|----------|
| 1 | **Self-Organizing Map (SOM)** | Unsupervised seismic facies classification |
| 2 | **Vision LLM (GPT-4o mini)** | Automatic geological interpretation of seismic images |
| 3 | **RAG (arXiv API)** | Scientific report enrichment with relevant literature |

### Datasets used
- `FlatVel_A` — flat horizontal layers, velocity gradient with depth only
- `FlatFault_A` — horizontal layers with vertical fault discontinuities
- `CurveVel_A` — curved layers (future work)

### Cost summary
- Vision LLM: ~$0.003 per sample (GPT-4o mini, `detail:high`)
- RAG: ~$0.0003 per sample (text only)
- **Total for 6 samples analyzed: ~$0.021**

---

## 0. Environment Setup

Standard Kaggle environment check. Lists all available input files.
The `kagglehub.dataset_download` call below is the Kaggle default template — not used in this notebook since data is already available via the competition input path.

In [ ]:
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 1. Dataset Exploration

### 1.1 Directory Structure

The OpenFWI competition data lives under `/kaggle/input/competitions/waveform-inversion/train_samples/`.
We explore the top-level structure to understand the file layout before loading anything.

In [ ]:
import os

base = '/kaggle/input/competitions/waveform-inversion'
for item in os.listdir(base):
    full_path = os.path.join(base, item)
    if os.path.isdir(full_path):
        files = os.listdir(full_path)
        print(f"📁 {item}/ — {len(files)} items")
        for f in files[:3]:
            print(f"     {f}")
    else:
        print(f"📄 {item}")

### 1.2 Shape and Range Inspection

We iterate over the three dataset families and inspect shapes and value ranges.

**Key observations:**
- `FlatVel_A` and `CurveVel_A`: organized as `data/` and `model/` subdirectories
- `FlatFault_A`: files named directly as `seis_*.npy` and `vel_*.npy`
- All seismic arrays: `(500, 5, 1000, 70)` — 500 samples, 5 sources, 1000 time steps, 70 receivers
- All velocity maps: `(500, 1, 70, 70)` — 500 samples, 70×70 pixel grid, range 1500–4500 m/s

In [ ]:
import numpy as np
import os

base = '/kaggle/input/competitions/waveform-inversion/train_samples'

for familia in ['FlatVel_A', 'FlatFault_A', 'CurveVel_A']:
    path = os.path.join(base, familia)
    items = os.listdir(path)
    print(f"\n📁 {familia}/")

    if os.path.isdir(os.path.join(path, items[0])):
        for sub in items:
            subpath = os.path.join(path, sub)
            files = os.listdir(subpath)
            arr = np.load(os.path.join(subpath, files[0]))
            print(f"   📁 {sub}/ — {len(files)} files")
            print(f"        Shape: {arr.shape} | dtype: {arr.dtype}")
            print(f"        Min: {arr.min():.4f} | Max: {arr.max():.4f}")
    else:
        seis = [f for f in items if f.startswith('seis')]
        vels = [f for f in items if f.startswith('vel')]
        print(f"   📄 seis: {len(seis)} files | vel: {len(vels)} files")
        if seis:
            arr = np.load(os.path.join(path, seis[0]))
            print(f"   seis shape: {arr.shape} | dtype: {arr.dtype}")
            print(f"   Min: {arr.min():.4f} | Max: {arr.max():.4f}")
        if vels:
            arr = np.load(os.path.join(path, vels[0]))
            print(f"   vel  shape: {arr.shape} | dtype: {arr.dtype}")
            print(f"   Min: {arr.min():.4f} | Max: {arr.max():.4f}")

### 1.3 Visual EDA — FlatVel_A

We load `FlatVel_A/data2.npy` and `FlatVel_A/model2.npy` for initial visual exploration.
The 7-panel figure shows all 5 seismic sources, the ground truth velocity map, and the velocity distribution histogram.

> **Note:** The variable names `data` and `model` are used here for the initial EDA only. 
> Later in the notebook they are superseded by `data_v`/`model_v` (FlatVel_A) and `data_f`/`model_f` (FlatFault_A) 
> to avoid accidental overwriting between datasets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

base = '/kaggle/input/competitions/waveform-inversion/train_samples'

data  = np.load(f'{base}/FlatVel_A/data/data2.npy')   # (500, 5, 1000, 70)
model = np.load(f'{base}/FlatVel_A/model/model2.npy') # (500, 1, 70, 70)

sample_idx = 0
fig, axes = plt.subplots(1, 7, figsize=(20, 5))

for i in range(5):
    axes[i].imshow(data[sample_idx, i], aspect='auto', cmap='seismic', vmin=-5, vmax=5)
    axes[i].set_title(f'Source {i+1}')
    axes[i].set_xlabel('Receivers')
    axes[i].set_ylabel('Time')

axes[5].imshow(model[sample_idx, 0], aspect='auto', cmap='jet', vmin=1500, vmax=4500)
axes[5].set_title('Vel Map (GT)')
axes[5].set_xlabel('X')
axes[5].set_ylabel('Depth')

axes[6].hist(model[sample_idx, 0].flatten(), bins=50, color='steelblue')
axes[6].set_title('Velocity distribution')
axes[6].set_xlabel('m/s')

plt.suptitle('FlatVel_A — Sample 0 | EDA', fontsize=14)
plt.tight_layout()
plt.savefig('eda_flatvel.png', dpi=150)
plt.show()
plt.close()

print(f"Total samples available: {data.shape[0] * 2} per family")

---
## 2. Layer 1 — Self-Organizing Map (SOM) for FlatVel_A

We train an unsupervised SOM to classify seismic facies from acoustic attributes.
This is analogous to the SEG-Y horizon attribute workflow used in real exploration.

### Feature engineering strategy

For `FlatVel_A`, velocity only varies with **depth** (horizontal layers). 
We extract features **per depth row** — one feature vector per depth level per sample:

```
For each depth d (0–69):
    map depth → time index (depth_to_time)
    extract snapshot at that time across all 70 receivers × 5 sources
    compute: mean, max, std, energy, zero-crossings  →  25 features
```

This gives a `(n_samples × 70, 25)` feature matrix — one row per depth horizon.

In [ ]:
!pip install minisom -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from minisom import MiniSom
from sklearn.preprocessing import StandardScaler

### 2.1 First attempt: features per receiver column (v1)

Initial approach: extract features per **receiver column X** (vertical traces).
This yielded vertical stripe patterns in the facies map — not geologically meaningful,
since FlatVel_A has horizontal layering, not vertical.

> ⚠️ **This version was superseded by v2.** Kept here to document the iteration process.

In [ ]:
base = '/kaggle/input/competitions/waveform-inversion/train_samples/FlatVel_A'
data  = np.load(f'{base}/data/data2.npy')
model = np.load(f'{base}/model/model2.npy')

print(f"Seismic : {data.shape}")
print(f"Vel map : {model.shape}")

N_SAMPLES = 100

def extract_features_v1(data, model, n_samples=100):
    """Features per receiver column X. Produces vertical stripes — not ideal for FlatVel_A."""
    features_list, vel_list = [], []
    for s in range(n_samples):
        seis = data[s]
        vmap = model[s, 0]
        for x in range(70):
            traces = seis[:, :, x]  # (5, 1000)
            feat = np.concatenate([
                traces.mean(axis=1), traces.max(axis=1),
                traces.std(axis=1), (traces**2).mean(axis=1),
                (np.diff(np.sign(traces)) != 0).sum(axis=1).astype(float)
            ])
            features_list.append(feat)
            vel_list.append(vmap[:, x].mean())
    return np.array(features_list), np.array(vel_list)

X, y_vel = extract_features_v1(data, model, n_samples=N_SAMPLES)
print(f"Features shape: {X.shape}")
print(f"Velocity range: {y_vel.min():.0f} – {y_vel.max():.0f} m/s")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

SOM_X, SOM_Y = 5, 5
som = MiniSom(SOM_X, SOM_Y, input_len=25, sigma=1.5, learning_rate=0.5, random_seed=42)
som.random_weights_init(X_scaled)
som.train_random(X_scaled, num_iteration=5000, verbose=True)

winners = np.array([som.winner(x) for x in X_scaled])
facies_ids = winners[:, 0] * SOM_Y + winners[:, 1]
print(f"\nUnique facies found: {len(np.unique(facies_ids))}")

### 2.2 Corrected approach: features per depth row (v2)

We switch to extracting features **per depth row** (horizontal horizons).
This aligns with the physical structure of FlatVel_A and produces horizontal band patterns
that correctly reflect the layered velocity model.

We test three SOM grid sizes:
- 5×5 (25 neurons) — too granular
- 3×3 (9 neurons) — too coarse
- **4×4 (16 neurons) — best balance** ✅

The final model `som2` / `scaler2` (QE ≈ 0.88) is used throughout the rest of the notebook.

In [ ]:
def extract_features_v2(data, model, n_samples=100):
    """
    Features per depth row — one vector per depth level per sample.
    Maps depth index (0-69) to time index (0-999) linearly.
    Returns X (n_samples*70, 25) and y_vel (n_samples*70,).
    """
    features_list, vel_list = [], []
    depth_to_time = np.linspace(0, 999, 70).astype(int)

    for s in range(n_samples):
        seis = data[s]       # (5, 1000, 70)
        vmap = model[s, 0]   # (70, 70)

        for d in range(70):
            t = depth_to_time[d]
            traces_at_t = seis[:, t, :]  # (5, 70)
            feat = np.concatenate([
                traces_at_t.mean(axis=1),
                traces_at_t.max(axis=1),
                traces_at_t.std(axis=1),
                (traces_at_t**2).mean(axis=1),
                np.abs(np.diff(np.sign(traces_at_t))).sum(axis=1).astype(float)
            ])
            features_list.append(feat)
            vel_list.append(vmap[d, :].mean())

    return np.array(features_list), np.array(vel_list)

X2, y_vel2 = extract_features_v2(data, model, n_samples=100)
print(f"Features shape: {X2.shape}")

scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2)

# Final SOM: 4x4 grid — best balance between granularity and geological meaning
som2 = MiniSom(4, 4, input_len=25, sigma=1.2, learning_rate=0.5, random_seed=42)
som2.random_weights_init(X2_scaled)
som2.train_random(X2_scaled, num_iteration=5000, verbose=True)

feats_s0 = X2_scaled[0*70 : 1*70]
winners_s0 = np.array([som2.winner(x) for x in feats_s0])
facies_s0 = winners_s0[:, 0] * 4 + winners_s0[:, 1]
facies_map2 = np.tile(facies_s0.reshape(-1, 1), (1, 70))

print(f"Unique facies active: {len(np.unique(facies_s0))} of 16 possible")

cmap = plt.colormaps['tab20'].resampled(16)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(data[0, 0], aspect='auto', cmap='seismic', vmin=-5, vmax=5)
axes[0].set_title('Seismic (Source 1)')

im1 = axes[1].imshow(model[0, 0], aspect='auto', cmap='jet', vmin=1500, vmax=4500)
axes[1].set_title('Vel Map Ground Truth (m/s)')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(facies_map2, aspect='auto', cmap=cmap, vmin=0, vmax=15)
axes[2].set_title(f'SOM Facies 4x4 ({len(np.unique(facies_s0))} active)')
plt.colorbar(im2, ax=axes[2])

plt.suptitle('FlatVel_A — Sample 0 | SOM 4x4 (final)', fontsize=14)
plt.tight_layout()
plt.savefig('som_facies_4x4.png', dpi=150)
plt.show()
plt.close()

### 2.3 Sample selection by velocity variability

To validate the SOM across diverse geological scenarios, we select 3 samples
spanning the full range of subsurface complexity:

- **Max variability** (idx=194, std≈1219 m/s) — many distinct velocity layers
- **Median variability** (idx=69, std≈648 m/s) — typical sample
- **Min variability** (idx=347, std≈92 m/s) — nearly homogeneous

This stratified selection tests whether the SOM correctly collapses to fewer facies
when the subsurface is geologically simple.

In [ ]:
vel_variability = []
for i in range(500):
    vmap = model[i, 0]
    variability = np.std(vmap.mean(axis=1))
    vel_variability.append(variability)

vel_variability = np.array(vel_variability)

idx_max = vel_variability.argmax()
idx_min = vel_variability.argmin()
idx_med = np.argsort(vel_variability)[len(vel_variability)//2]

print(f"Highest variability : idx={idx_max} | std={vel_variability[idx_max]:.1f} m/s")
print(f"Median variability  : idx={idx_med} | std={vel_variability[idx_med]:.1f} m/s")
print(f"Lowest variability  : idx={idx_min} | std={vel_variability[idx_min]:.1f} m/s")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, idx, label in zip(axes,
                           [idx_max, idx_med, idx_min],
                           ['High variability', 'Median', 'Low variability']):
    im = ax.imshow(model[idx, 0], aspect='auto', cmap='jet', vmin=1500, vmax=4500)
    ax.set_title(f'{label}\nidx={idx} | std={vel_variability[idx]:.1f} m/s')
    ax.set_xlabel('X'); ax.set_ylabel('Depth')
    plt.colorbar(im, ax=ax)

plt.suptitle('Sample selection for SOM validation', fontsize=13)
plt.tight_layout()
plt.show()
plt.close()

### 2.4 SOM validation on 3 selected samples

In [ ]:
test_samples = [
    (194, 'High variability',  1218.9),
    (69,  'Median',             648.0),
    (347, 'Low variability',     92.1),
]

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
cmap_facies = plt.colormaps['tab20'].resampled(16)
depth_to_time = np.linspace(0, 999, 70).astype(int)

for row, (idx, label, std) in enumerate(test_samples):
    seis = data[idx]
    vmap = model[idx, 0]

    feats = []
    for d in range(70):
        t = depth_to_time[d]
        traces_at_t = seis[:, t, :]
        feat = np.concatenate([
            traces_at_t.mean(axis=1), traces_at_t.max(axis=1),
            traces_at_t.std(axis=1), (traces_at_t**2).mean(axis=1),
            np.abs(np.diff(np.sign(traces_at_t))).sum(axis=1).astype(float)
        ])
        feats.append(feat)

    feats_scaled = scaler2.transform(np.array(feats))
    winners = np.array([som2.winner(x) for x in feats_scaled])
    facies  = winners[:, 0] * 4 + winners[:, 1]
    facies_map = np.tile(facies.reshape(-1, 1), (1, 70))
    n_unique = len(np.unique(facies))

    axes[row, 0].imshow(seis[0], aspect='auto', cmap='seismic', vmin=-5, vmax=5)
    axes[row, 0].set_title(f'Seismic — {label} (idx={idx})')

    im1 = axes[row, 1].imshow(vmap, aspect='auto', cmap='jet', vmin=1500, vmax=4500)
    axes[row, 1].set_title(f'Vel Map | std={std:.0f} m/s')
    plt.colorbar(im1, ax=axes[row, 1])

    im2 = axes[row, 2].imshow(facies_map, aspect='auto', cmap=cmap_facies, vmin=0, vmax=15)
    axes[row, 2].set_title(f'SOM Facies | {n_unique} active of 16')
    plt.colorbar(im2, ax=axes[row, 2])

    print(f"idx={idx:3d} | {label:20s} | active facies: {n_unique} | {np.unique(facies)}")

plt.suptitle('SOM Validation — 3 extreme samples | FlatVel_A', fontsize=14)
plt.tight_layout()
plt.savefig('som_validation_3samples.png', dpi=150)
plt.show()
plt.close()

### 2.5 Homogeneity threshold

For nearly-homogeneous samples (velocity std < 200 m/s), the SOM may assign
multiple facies to what is effectively one geological unit.
We add a threshold: if `std(vel_profile) < threshold`, assign a single facies.

In [ ]:
def predict_facies(seis, vmap, som, scaler, threshold_std=200):
    """Predict SOM facies with homogeneity threshold."""
    depth_to_time = np.linspace(0, 999, 70).astype(int)
    sample_std = np.std(vmap.mean(axis=1))

    feats = []
    for d in range(70):
        t = depth_to_time[d]
        traces_at_t = seis[:, t, :]
        feat = np.concatenate([
            traces_at_t.mean(axis=1), traces_at_t.max(axis=1),
            traces_at_t.std(axis=1), (traces_at_t**2).mean(axis=1),
            np.abs(np.diff(np.sign(traces_at_t))).sum(axis=1).astype(float)
        ])
        feats.append(feat)

    feats_scaled = scaler.transform(np.array(feats))

    if sample_std < threshold_std:
        facies = np.zeros(70, dtype=int)
        print(f"  → Homogeneous (std={sample_std:.1f}) — single facies assigned")
    else:
        winners = np.array([som.winner(x) for x in feats_scaled])
        facies  = winners[:, 0] * 4 + winners[:, 1]
        print(f"  → std={sample_std:.1f} m/s | {len(np.unique(facies))} active facies")

    return np.tile(facies.reshape(-1, 1), (1, 70))

for idx, label, std in test_samples:
    print(f"\n{label} (idx={idx})")
    fm = predict_facies(data[idx], model[idx, 0], som2, scaler2)
    print(f"  Final unique facies: {len(np.unique(fm))}")

### 2.6 Lithological interpretation

We map SOM facies to lithological classes using the velocity ranges defined in `VEL_RANGES`.
Each facies is assigned to a lithology class based on its mean velocity.

The `vel_to_litho` function implements this mapping.

In [ ]:
# Geological velocity ranges — physically meaningful classification
VEL_RANGES = [
    (1500, 2000, 'Soft sediments',      '#4575b4'),
    (2000, 2500, 'Compacted sediments', '#74add1'),
    (2500, 3000, 'Sedimentary rock',    '#abd9e9'),
    (3000, 3500, 'Hard rock',           '#fdae61'),
    (3500, 4500, 'Very hard rock',      '#d73027'),
]

def vel_to_litho(vel):
    """Map velocity (m/s) to lithology class index, label, and color."""
    for i, (vmin, vmax, label, color) in enumerate(VEL_RANGES):
        if vmin <= vel < vmax:
            return i, label, color
    return len(VEL_RANGES)-1, VEL_RANGES[-1][2], VEL_RANGES[-1][3]

print("vel_to_litho() defined ✅")
print("VEL_RANGES:")
for vmin, vmax, label, color in VEL_RANGES:
    print(f"  {vmin}–{vmax} m/s → {label} ({color})")

---
## 3. Layer 1 — SOM for FlatFault_A

FlatFault_A introduces **vertical fault discontinuities** on top of horizontal layering.
The velocity model is no longer purely 1D — faults create lateral variations.

### Key difference from FlatVel_A

| | FlatVel_A | FlatFault_A |
|--|-----------|-------------|
| SOM type | 1D (per depth row) | **2D (per pixel)** |
| Features | 25 (5 sources × 5 attrs) | **7 (point-local attrs)** |
| Grid | 4×4 | 4×4 |
| Input len | 25 | **7** |
| QE | 0.88 | **0.44** |

The 2D pixel-wise approach captures lateral discontinuities that the 1D row-wise approach would miss.

In [ ]:
import os

base_fault = '/kaggle/input/competitions/waveform-inversion/train_samples/FlatFault_A'
files = os.listdir(base_fault)
seis_files = sorted([f for f in files if f.startswith('seis')])
vel_files  = sorted([f for f in files if f.startswith('vel')])

print("Seismic files:", seis_files)
print("Vel files:    ", vel_files)

data_f  = np.load(f'{base_fault}/{seis_files[0]}')
model_f = np.load(f'{base_fault}/{vel_files[0]}')

print(f"\nSeismic shape : {data_f.shape}")
print(f"Vel map shape : {model_f.shape}")

# Quick visual — 3 samples to see fault variation
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for col, idx in enumerate([0, 100, 400]):
    axes[0, col].imshow(data_f[idx, 0], aspect='auto', cmap='seismic', vmin=-5, vmax=5)
    axes[0, col].set_title(f'Seismic idx={idx}')

    im1 = axes[1, col].imshow(model_f[idx, 0], aspect='auto', cmap='jet', vmin=1500, vmax=4500)
    axes[1, col].set_title(f'Vel Map idx={idx}')
    plt.colorbar(im1, ax=axes[1, col], label='m/s')

plt.suptitle('FlatFault_A — Initial exploration (idx 0, 100, 400)', fontsize=13)
plt.tight_layout()
plt.show()
plt.close()

### 3.1 Feature extraction — 2D pixel-wise

For each pixel `(depth d, lateral x)` in the 70×70 velocity map, we extract
7 local features from the 5-source seismic snapshot at the corresponding time:

```
vals = seis[:, t, x]  # 5 values (one per source)
features = [mean, max, min, std, energy, range, zero_crossings]
```

This produces `n_samples × 70 × 70 = 245,000` feature vectors for 50 training samples.

In [ ]:
def extract_features_2d(data_arr, model_arr, n_samples=100):
    """
    Pixel-wise feature extraction for FlatFault_A.
    Returns X (n*70*70, 7), y (n*70*70,), coords list.
    """
    depth_to_time = np.linspace(0, 999, 70).astype(int)
    features_list, vel_list, coords = [], [], []

    for s in range(n_samples):
        seis = data_arr[s]
        vmap = model_arr[s, 0]
        for d in range(70):
            t = depth_to_time[d]
            for x in range(70):
                vals = seis[:, t, x]  # (5,)
                feat = np.array([
                    vals.mean(), vals.max(), vals.min(), vals.std(),
                    (vals**2).mean(),
                    vals.max() - vals.min(),
                    np.abs(np.diff(np.sign(vals))).sum()
                ])
                features_list.append(feat)
                vel_list.append(vmap[d, x])
                coords.append((s, d, x))

    return np.array(features_list), np.array(vel_list), coords

print("Extracting 2D features... (1-2 min)")
X_f, y_f, coords_f = extract_features_2d(data_f, model_f, n_samples=50)
print(f"Features shape : {X_f.shape}")
print(f"Vel range      : {y_f.min():.0f} – {y_f.max():.0f} m/s")

scaler_f = StandardScaler()
X_f_scaled = scaler_f.fit_transform(X_f)

som_f = MiniSom(4, 4, input_len=7, sigma=1.2, learning_rate=0.5, random_seed=42)
som_f.random_weights_init(X_f_scaled)
print("Training FlatFault_A SOM...")
som_f.train_random(X_f_scaled, num_iteration=10000, verbose=True)
print(f"Quantization error: {som_f.quantization_error(X_f_scaled):.4f}")

### 3.2 Visualization — FlatFault_A lithology maps

Three representative samples:
- **idx=0**: No fault (flat horizontal layers only)
- **idx=100**: Central fault (clear vertical discontinuity)
- **idx=400**: Complex fault geometry

The `plot_fault_sample` function generates a 4-panel figure:
seismic | vel map | SOM lithology 2D | geological legend

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

def plot_fault_sample(idx, label, data_arr, model_arr, som, scaler):
    seis = data_arr[idx]
    vmap = model_arr[idx, 0]
    depth_to_time = np.linspace(0, 999, 70).astype(int)

    feats = []
    for d in range(70):
        t = depth_to_time[d]
        for x in range(70):
            vals = seis[:, t, x]
            feat = np.array([
                vals.mean(), vals.max(), vals.min(), vals.std(),
                (vals**2).mean(), vals.max()-vals.min(),
                np.abs(np.diff(np.sign(vals))).sum()
            ])
            feats.append(feat)

    feats_scaled = scaler.transform(np.array(feats))
    winners      = np.array([som.winner(x) for x in feats_scaled])
    facies_flat  = winners[:, 0] * 4 + winners[:, 1]
    facies_map   = facies_flat.reshape(70, 70)

    facies_vel = {}
    for d in range(70):
        for x in range(70):
            f = facies_map[d, x]
            if f not in facies_vel:
                facies_vel[f] = []
            facies_vel[f].append(vmap[d, x])
    facies_vel = {f: np.mean(v) for f, v in facies_vel.items()}

    litho_map = np.zeros((70, 70), dtype=int)
    for f, vel in facies_vel.items():
        clase, _, _ = vel_to_litho(vel)
        litho_map[facies_map == f] = clase

    colors     = [r[3] for r in VEL_RANGES]
    cmap_litho = ListedColormap(colors)
    norm_litho = BoundaryNorm(boundaries=range(len(VEL_RANGES)+1), ncolors=len(VEL_RANGES))

    fig = plt.figure(figsize=(22, 6))
    gs  = fig.add_gridspec(1, 4, width_ratios=[3, 3, 3, 2.5], wspace=0.38)
    ax0, ax1, ax2, ax3 = [fig.add_subplot(gs[i]) for i in range(4)]

    ax0.imshow(seis[0], aspect='auto', cmap='seismic', vmin=-5, vmax=5)
    ax0.set_title('Seismic (Source 1)', fontsize=11, pad=8)
    ax0.set_xlabel('Receivers'); ax0.set_ylabel('Time')

    im1 = ax1.imshow(vmap, aspect='auto', cmap='jet', vmin=1500, vmax=4500)
    ax1.set_title('Vel Map Ground Truth', fontsize=11, pad=8)
    ax1.set_xlabel('X'); ax1.set_ylabel('Depth')
    plt.colorbar(im1, ax=ax1, label='m/s', fraction=0.046, pad=0.04)

    clases_unicas = np.unique(litho_map)
    im2 = ax2.imshow(litho_map, aspect='auto', cmap=cmap_litho, norm=norm_litho)
    ax2.set_title(f'SOM Lithology 2D\n{len(clases_unicas)} class(es) detected', fontsize=11, pad=8)
    ax2.set_xlabel('X'); ax2.set_ylabel('Depth')

    ax3.axis('off')
    ax3.set_title('Lithology → Velocity', fontsize=11, pad=10)
    present = set(litho_map.flatten())
    y_pos   = np.linspace(0.88, 0.08, len(VEL_RANGES))

    for i, (vmin, vmax, desc, color) in enumerate(VEL_RANGES):
        y = y_pos[i]
        alpha = 1.0 if i in present else 0.25
        ax3.add_patch(plt.Rectangle((0.02, y-0.045), 0.14, 0.08,
                                     color=color, alpha=alpha, transform=ax3.transAxes, clip_on=False))
        ax3.text(0.20, y+0.015, f'{vmin}–{vmax} m/s', transform=ax3.transAxes,
                 fontsize=9.5, va='center', fontfamily='monospace', alpha=alpha)
        ax3.text(0.20, y-0.022, desc, transform=ax3.transAxes,
                 fontsize=8.5, va='center', color='#444444', style='italic', alpha=alpha)

    ax3.text(0.02, 0.0, '* faded = absent in this sample',
             transform=ax3.transAxes, fontsize=7.5, color='gray', style='italic')

    std_val = np.std(vmap.mean(axis=1))
    fig.suptitle(f'{label}   |   idx={idx}   |   vel std = {std_val:.0f} m/s', fontsize=13, y=1.02)
    plt.savefig(f'fault_litho_{idx}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"✅ Saved: fault_litho_{idx}.png")

for idx, label in [(0,   'FlatFault_A — no fault'),
                   (100, 'FlatFault_A — central fault'),
                   (400, 'FlatFault_A — complex fault')]:
    plot_fault_sample(idx, label, data_f, model_f, som_f, scaler_f)

---
## 4. Layer 2 — Vision LLM: Connectivity Tests

Before committing to a Vision LLM provider, we tested several free options from Kaggle.
This section documents the failed attempts — included for transparency and reproducibility.

### HuggingFace token setup

In [ ]:
!pip install huggingface_hub -q

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

from huggingface_hub import whoami
info = whoami(token=HF_TOKEN)
print(f"✅ Connected as: {info['name']}")

### 4.1 Connectivity test — which endpoints are reachable from Kaggle?

Kaggle's network blocks `api-inference.huggingface.co` but allows `huggingface.co` (Spaces).
We probe each endpoint before attempting any API calls.

In [ ]:
import requests

urls = [
    'https://api-inference.huggingface.co',  # ❌ blocked by Kaggle
    'https://huggingface.co',               # ✅ accessible
    'https://www.google.com',               # ✅ accessible
    'https://api.openai.com',               # ✅ accessible
    'https://export.arxiv.org',             # ✅ accessible
]

for url in urls:
    try:
        r = requests.get(url, timeout=5)
        print(f"✅ {url} → {r.status_code}")
    except Exception as e:
        print(f"❌ {url} → {str(e)[:60]}")

### 4.2 Failed attempts — HuggingFace Gradio Spaces

We tried four Vision models via Gradio Spaces. All failed for different reasons:

| Model | Error | Reason |
|-------|-------|--------|
| `vikhyatk/moondream2` | upstream error | Space overloaded |
| `microsoft/Florence-2` | 401 | Auth required |
| `qnguyen3/nanollava` | RUNTIME_ERROR | Space broken |
| `Salesforce/BLIP2` | 429 | Rate limited |

> ⚠️ The cells below are kept for documentation. They will fail — this is expected.

In [ ]:
from gradio_client import Client, handle_file

# Attempt 1 — Moondream2
# Result: upstream error (Space overloaded or endpoint changed)
try:
    c = Client("vikhyatk/moondream2")
    print("Connected. Endpoints:", c.view_api())
except Exception as e:
    print(f"❌ Moondream2: {str(e)[:120]}")

In [ ]:
# Attempt 2 — Florence-2, nanollava, BLIP2 via Gradio
# Results: 401, RUNTIME_ERROR, 429 respectively
spaces_to_try = [
    ("microsoft/Florence-2-large", "/process_image"),
    ("qnguyen3/nanollava",          "/chat"),
    ("Salesforce/BLIP2",            "/predict"),
]

for space, api in spaces_to_try:
    try:
        c = Client(space)
        print(f"✅ Connected to {space}")
        break
    except Exception as e:
        print(f"❌ {space}: {str(e)[:80]}")

In [ ]:
# Attempt 3 — BLIP2 via direct HTTP (bypassing gradio_client websocket bug)
# Result: 429 Too Many Requests — public Space rate limited
import requests, json

# NOTE: img_b64 must be defined from a previous cell
url = "https://salesforce-blip2.hf.space/run/predict"
payload = {
    "fn_index": 2,
    "data": [
        f"data:image/png;base64,{img_b64}",
        "What geological structures are visible in this seismic velocity map?"
    ]
}

r = requests.post(url, json=payload, timeout=90)
print(f"Status: {r.status_code}")
if r.status_code == 200:
    print(r.json()['data'])
else:
    print(r.text[:300])  # Expected: 429 or HTML error page

### 4.3 Solution — OpenAI GPT-4o mini Vision

`api.openai.com` is accessible from Kaggle. GPT-4o mini with `detail:high` provides
genuine image understanding at low cost (~$0.003/call).

**Cost analysis:**
- `detail:low` → ~2,988 tokens → $0.00072 — too generic, doesn't actually see the image
- `detail:high` + large PNG → ~48,409 tokens → $0.00737 — accurate but expensive
- `detail:high` + JPEG 1200px → **~20,000 tokens → $0.003** ✅ — best balance

**Key optimization:** convert matplotlib figure → PNG → PIL resize → JPEG quality=82
before encoding to base64. This reduces tokens 2.4× vs. sending raw PNG.

In [ ]:
# Connectivity test
import requests
try:
    r = requests.get("https://api.openai.com", timeout=5)
    print(f"✅ OpenAI accessible → {r.status_code}")
except Exception as e:
    print(f"❌ {str(e)[:80]}")

OPENAI_KEY = secrets.get_secret("OPENAI_API_KEY")
print("OPENAI_KEY loaded ✅")

---
## 5. Layer 2 — Vision LLM: Production Functions

### 5.1 make_seismic_plot

Generates the 3-panel figure sent to the Vision LLM:
- LEFT: seismic waveform (source 1)
- CENTER: velocity map (ground truth)
- RIGHT: SOM lithology classification

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

def make_seismic_plot(idx, data_arr, model_arr, litho_map):
    """
    Generates 3-panel seismic figure for Vision LLM input.
    Returns matplotlib figure (not shown, not saved).
    """
    colors     = [r[3] for r in VEL_RANGES]
    cmap_litho = ListedColormap(colors)
    norm_litho = BoundaryNorm(boundaries=range(len(VEL_RANGES)+1), ncolors=len(VEL_RANGES))

    fig = plt.figure(figsize=(16, 5))
    gs  = fig.add_gridspec(1, 3, wspace=0.35)
    ax0, ax1, ax2 = [fig.add_subplot(gs[i]) for i in range(3)]

    ax0.imshow(data_arr[idx, 0], aspect='auto', cmap='seismic', vmin=-5, vmax=5)
    ax0.set_title('Seismic Data (Source 1)', fontsize=11)
    ax0.set_xlabel('Receivers'); ax0.set_ylabel('Time')

    im1 = ax1.imshow(model_arr[idx, 0], aspect='auto', cmap='jet', vmin=1500, vmax=4500)
    ax1.set_title('Velocity Map (m/s)', fontsize=11)
    ax1.set_xlabel('X'); ax1.set_ylabel('Depth')
    plt.colorbar(im1, ax=ax1, label='m/s', fraction=0.046)

    im2 = ax2.imshow(litho_map, aspect='auto', cmap=cmap_litho, norm=norm_litho)
    ax2.set_title('SOM Lithology Classification', fontsize=11)
    ax2.set_xlabel('X'); ax2.set_ylabel('Depth')

    legend_labels = [f'{vmin}–{vmax} m/s: {desc}' for vmin, vmax, desc, _ in VEL_RANGES]
    handles = [plt.Rectangle((0,0),1,1, color=c) for _,_,_,c in VEL_RANGES]
    ax2.legend(handles, legend_labels, loc='lower right', fontsize=7, framealpha=0.85)

    fig.suptitle(f'Seismic Analysis — idx={idx}', fontsize=13)
    return fig

print("make_seismic_plot() defined ✅")

### 5.2 compute_litho_map — FlatFault_A

Computes the SOM lithology map for any FlatFault_A sample.
Uses the 2D pixel-wise SOM (`som_f`, `scaler_f`).

In [ ]:
def compute_litho_map(idx, data_arr, model_arr, som, scaler):
    """
    Computes 2D lithology map for FlatFault_A using pixel-wise SOM.
    Returns: litho_map (70,70), facies_map (70,70), vmap (70,70).
    """
    seis = data_arr[idx]
    vmap = model_arr[idx, 0]
    depth_to_time = np.linspace(0, 999, 70).astype(int)

    feats = []
    for d in range(70):
        t = depth_to_time[d]
        for x in range(70):
            vals = seis[:, t, x]
            feats.append(np.array([
                vals.mean(), vals.max(), vals.min(), vals.std(),
                (vals**2).mean(), vals.max()-vals.min(),
                np.abs(np.diff(np.sign(vals))).sum()
            ]))

    feats_scaled = scaler.transform(np.array(feats))
    winners      = np.array([som.winner(f) for f in feats_scaled])
    facies_map   = (winners[:, 0] * 4 + winners[:, 1]).reshape(70, 70)

    facies_vel = {}
    for d in range(70):
        for x in range(70):
            f = facies_map[d, x]
            if f not in facies_vel:
                facies_vel[f] = []
            facies_vel[f].append(vmap[d, x])
    facies_vel = {f: np.mean(v) for f, v in facies_vel.items()}

    litho_map = np.zeros((70, 70), dtype=int)
    for f, vel in facies_vel.items():
        clase, _, _ = vel_to_litho(vel)
        litho_map[facies_map == f] = clase

    return litho_map, facies_map, vmap

print("compute_litho_map() defined ✅")

### 5.3 detect_fault_from_som

Numerically detects fault presence from the SOM lithology map.
Computes mean lateral difference across the lithology map — 
high scores indicate abrupt horizontal changes consistent with faulting.

This result is injected into the Vision LLM prompt as prior context,
improving fault detection accuracy significantly (especially for no-fault samples).

In [ ]:
def detect_fault_from_som(litho_map):
    """
    Detects lateral discontinuity in SOM lithology map.
    Returns: has_fault (bool), fault_score (float), fault_x (int).
    """
    lateral_diff = np.diff(litho_map, axis=1)  # (70, 69)
    fault_score  = np.abs(lateral_diff).mean()
    col_changes  = np.abs(lateral_diff).sum(axis=0)
    fault_x      = col_changes.argmax()
    has_fault    = fault_score > 0.3  # empirical threshold

    return {
        'has_fault':   has_fault,
        'fault_score': round(float(fault_score), 3),
        'fault_x':     int(fault_x)
    }

print("detect_fault_from_som() defined ✅")

### 5.4 analyze_seismic_vision — main Vision LLM function

Full pipeline:
1. matplotlib figure → PNG → PIL resize (max 1200px) → JPEG q=82 → base64
2. Build prompt with SOM fault context injected
3. Call GPT-4o mini with `detail:high`
4. Return interpretation text + token usage + cost

In [ ]:
from PIL import Image
import io, base64, requests

def analyze_seismic_vision(fig, openai_key, litho_map=None, prompt=None, max_width=1200):
    """
    Converts matplotlib figure to optimized JPEG and calls GPT-4o mini Vision.
    
    Args:
        fig: matplotlib Figure object
        openai_key: OpenAI API key string
        litho_map: optional (70,70) array — enables SOM fault context injection
        prompt: optional custom prompt (uses default if None)
        max_width: maximum image width in pixels before downsampling
    
    Returns:
        dict with keys: text, prompt_tokens, completion_tokens, cost_usd
    """
    # Step 1 — figure → optimized JPEG base64
    buf_raw = io.BytesIO()
    fig.savefig(buf_raw, format='png', dpi=120, bbox_inches='tight')
    buf_raw.seek(0)

    img_pil = Image.open(buf_raw)
    w, h = img_pil.size
    if w > max_width:
        img_pil = img_pil.resize((max_width, int(h * max_width / w)), Image.LANCZOS)

    buf = io.BytesIO()
    img_pil.convert('RGB').save(buf, format='JPEG', quality=82)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)  # avoid memory warning for >20 open figures

    # Step 2 — SOM context injection
    som_context = ""
    if litho_map is not None:
        fault_info = detect_fault_from_som(litho_map)
        som_context = (
            f"CONTEXT FROM SOM ANALYSIS: "
            f"{'Fault likely present' if fault_info['has_fault'] else 'NO fault detected'} "
            f"(discontinuity score={fault_info['fault_score']}, "
            f"most likely at lateral index {fault_info['fault_x']}).\n\n"
        )

    # Step 3 — prompt
    if prompt is None:
        prompt = f"""{som_context}You are an expert geophysicist. Analyze this image carefully.
THREE panels: LEFT=seismic waveform, CENTER=velocity map (m/s), RIGHT=SOM lithology.

Answer based ONLY on what you actually see. Be conservative about faults —
only report one if there is a CLEAR offset or discontinuity, not just noise.
This is a synthetic dataset (OpenFWI FlatFault_A): some samples have faults, others do not.

NOTE: Axes show pixel indices (0–70), NOT metric units. Use 'depth index' / 'lateral index'.

1. FAULT: Clear discontinuity in CENTER panel? YES/NO. If yes: lateral index and depth index.
2. VELOCITY RANGE: Min and max from colorbar.
3. LAYERS: How many distinct horizontal layers? Approximate depth index ranges.
4. SOM MAP: Dominant colors in top half vs bottom half of RIGHT panel.
5. SEISMIC: Reflection hyperbolas or offsets in LEFT panel?"""

    # Step 4 — API call
    r = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {openai_key}", "Content-Type": "application/json"},
        json={
            "model": "gpt-4o-mini",
            "max_tokens": 500,
            "messages": [{
                "role": "user",
                "content": [
                    {"type": "image_url",
                     "image_url": {"url": f"data:image/jpeg;base64,{img_b64}", "detail": "high"}},
                    {"type": "text", "text": prompt}
                ]
            }]
        },
        timeout=60
    )

    if r.status_code != 200:
        return {"error": r.text[:200]}

    resp  = r.json()
    usage = resp['usage']
    cost  = (usage['prompt_tokens'] * 0.00015 + usage['completion_tokens'] * 0.0006) / 1000

    return {
        "text":              resp['choices'][0]['message']['content'],
        "prompt_tokens":     usage['prompt_tokens'],
        "completion_tokens": usage['completion_tokens'],
        "cost_usd":          round(cost, 5)
    }

print("analyze_seismic_vision() ready ✅")

---
## 6. Layer 3 — RAG with arXiv

The third layer enriches the Vision LLM interpretation with scientific literature.

### Architecture

```
Vision LLM output text
    → extract_geo_keywords()  (keyword detection from text)
    → search_arxiv()          (arXiv Atom API, no key required)
    → build context string    (title + abstract, 400 chars each)
    → GPT-4o mini (text only) (synthesize report with citations)
    → geological report
```

**Why arXiv over a local FAISS index?**
- No installation, no embedding model, no disk space
- Always up-to-date papers
- Free, no API key
- Sufficient for this prototype — FAISS would be Layer 3 v2

In [ ]:
# Connectivity test
import requests
try:
    r = requests.get("https://export.arxiv.org/api/query?search_query=seismic&max_results=1", timeout=10)
    print(f"✅ arXiv accessible → {r.status_code}")
    print(r.text[:150])
except Exception as e:
    print(f"❌ {str(e)[:80]}")

In [ ]:
import xml.etree.ElementTree as ET

def extract_geo_keywords(interpretation_text):
    """
    Extracts geophysical keywords from Vision LLM output text.
    Keywords are tuned to retrieve FWI and deep-learning seismic papers.
    """
    keywords = []
    text_lower = interpretation_text.lower()

    if 'fault' in text_lower:
        keywords.append('fault detection full waveform inversion')
    if 'layer' in text_lower:
        keywords.append('seismic velocity model building synthetic')
    if 'velocity' in text_lower:
        keywords.append('FWI velocity estimation neural network')

    keywords.append('OpenFWI seismic benchmark deep learning')
    return keywords[:3]


def search_arxiv(keywords, max_results=4):
    """Searches arXiv Atom API. Returns list of dicts: title, authors, summary, link."""
    query = ' AND '.join(f'all:{kw}' for kw in keywords)
    params = {'search_query': query, 'max_results': max_results,
              'sortBy': 'relevance', 'sortOrder': 'descending'}

    r = requests.get("https://export.arxiv.org/api/query", params=params, timeout=15)
    if r.status_code != 200:
        return []

    ns   = {'atom': 'http://www.w3.org/2005/Atom'}
    root = ET.fromstring(r.text)
    papers = []

    for entry in root.findall('atom:entry', ns):
        papers.append({
            'title':   entry.find('atom:title',   ns).text.strip().replace('\n', ' '),
            'summary': entry.find('atom:summary', ns).text.strip().replace('\n', ' ')[:400],
            'link':    entry.find('atom:id',      ns).text.strip(),
            'authors': [a.find('atom:name', ns).text
                        for a in entry.findall('atom:author', ns)][:3]
        })
    return papers


def rag_enrich_interpretation(interpretation, openai_key, verbose=True):
    """
    Full RAG pipeline:
    1. Extract keywords from Vision LLM output
    2. Search arXiv for relevant papers
    3. Build context and call GPT-4o mini (text only)
    4. Return geological report with citations
    
    Cost: ~$0.0003 per call (text only, no image).
    """
    keywords = extract_geo_keywords(interpretation)
    if verbose:
        print(f"🔍 Keywords: {keywords}")

    papers = search_arxiv(keywords, max_results=4)
    if verbose:
        print(f"📄 Papers found: {len(papers)}")
        for p in papers:
            print(f"   • {p['title'][:70]}...")

    if not papers:
        return {"error": "No papers found", "interpretation": interpretation}

    context = "\n\n".join([
        f"[{i+1}] {p['title']}\nAuthors: {', '.join(p['authors'])}\nAbstract: {p['summary']}"
        for i, p in enumerate(papers)
    ])

    rag_prompt = f"""You are an expert geophysicist writing a scientific interpretation report.

SEISMIC IMAGE INTERPRETATION (from Vision AI):
{interpretation}

RELEVANT LITERATURE (from arXiv):
{context}

Write a concise geological report (200-250 words) that:
1. Confirms or refines the structural interpretation with scientific context
2. Cites at least 2 papers above as [1], [2], etc.
3. Suggests the best FWI approach for this subsurface
4. Identifies the most likely geological scenario

IMPORTANT: Use only 'depth index' and 'lateral index' — no metric units.
This is a synthetic dataset (OpenFWI) with pixel coordinates, not real meters."""

    r = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {openai_key}", "Content-Type": "application/json"},
        json={"model": "gpt-4o-mini", "max_tokens": 400,
              "messages": [{"role": "user", "content": rag_prompt}]},
        timeout=60
    )

    resp  = r.json()
    usage = resp['usage']
    cost  = (usage['prompt_tokens'] * 0.00015 + usage['completion_tokens'] * 0.0006) / 1000

    if verbose:
        print(f"\n💰 RAG cost: ${cost:.5f} | Tokens: {usage['total_tokens']}")

    return {
        "report":   resp['choices'][0]['message']['content'],
        "papers":   papers,
        "keywords": keywords,
        "cost_usd": round(cost, 5)
    }

print("✅ RAG pipeline ready")

---
## 7. Full Pipeline — FlatFault_A (3 samples)

We run the complete SOM → Vision LLM → RAG pipeline on three FlatFault_A samples:

| Sample | Expected | SOM fault score |
|--------|----------|-----------------|
| idx=0  | No fault | low score → NO fault context |
| idx=100 | Central fault | high score → fault context |
| idx=400 | Complex fault | medium score |

**Result:** idx=0 correctly reported "absence of significant faulting" after adding SOM context.
Without context, all three samples reported faults — demonstrating the value of the SOM bridge layer.

In [ ]:
samples = [
    (0,   'FlatFault_A — no fault'),
    (100, 'FlatFault_A — central fault'),
    (400, 'FlatFault_A — complex fault'),
]

all_results = {}

for idx, label in samples:
    print(f"\n{'='*60}")
    print(f"📍 {label} (idx={idx})")
    print('='*60)

    # Layer 1 — SOM lithology map
    litho_map, _, _ = compute_litho_map(idx, data_f, model_f, som_f, scaler_f)
    fig = make_seismic_plot(idx, data_f, model_f, litho_map)

    # Layer 2 — Vision LLM with SOM context
    print("🔭 Vision LLM...")
    vision = analyze_seismic_vision(fig, OPENAI_KEY, litho_map=litho_map)
    print(f"   ${vision['cost_usd']} | {vision['prompt_tokens']} tokens")

    # Layer 3 — RAG
    print("📚 RAG arXiv...")
    rag = rag_enrich_interpretation(vision['text'], OPENAI_KEY, verbose=False)
    print(f"   ${rag['cost_usd']} | {len(rag['papers'])} papers")

    total = vision['cost_usd'] + rag['cost_usd']
    all_results[idx] = {'label': label, 'vision': vision['text'],
                        'report': rag['report'], 'papers': rag['papers'], 'cost': total}

    print(f"\n{rag['report']}")
    print("\n📎 References:")
    for i, p in enumerate(rag['papers']):
        print(f"   [{i+1}] {p['title'][:65]}...")
    print(f"\n💰 Sample cost: ${total:.5f}")

print(f"\n{'='*60}")
total_fault = sum(r['cost'] for r in all_results.values())
print(f"FlatFault_A TOTAL: ${total_fault:.5f}")

---
## 8. FlatVel_A — Adapting the Pipeline

FlatVel_A has no faults. We adapt the pipeline with:
1. `compute_litho_map_vel()` — uses 1D SOM (`som2`/`scaler2`), features per depth row
2. `detect_gradient_from_som()` — replaces fault detection with velocity gradient analysis
3. `analyze_seismic_vision_vel()` — prompt adapted for layered compaction scenario

### Variable naming note
> ⚠️ During the notebook session, `model` was accidentally overwritten as a string 
> by a cell that used `model` as a generic variable name. 
> We reload FlatVel_A data as `data_v` / `model_v` to avoid collisions.

In [ ]:
# Reload FlatVel_A with unambiguous variable names
base = '/kaggle/input/competitions/waveform-inversion/train_samples/FlatVel_A'
data_v  = np.load(f'{base}/data/data2.npy')   # (500, 5, 1000, 70)
model_v = np.load(f'{base}/model/model2.npy') # (500, 1, 70, 70)

print(f"data_v  : {data_v.shape}  | {type(data_v)}")
print(f"model_v : {model_v.shape} | {type(model_v)}")

In [ ]:
def compute_litho_map_vel(idx, data_arr, model_arr, som, scaler):
    """
    Computes lithology map for FlatVel_A using the 1D row-wise SOM (som2/scaler2).
    Returns: litho_map (70,70), facies_1d (70,), vmap (70,70).
    """
    seis = data_arr[idx]
    vmap = model_arr[idx, 0]
    depth_to_time = np.linspace(0, 999, 70).astype(int)

    feats = []
    for d in range(70):
        t = depth_to_time[d]
        traces_at_t = seis[:, t, :]
        feat = np.concatenate([
            traces_at_t.mean(axis=1), traces_at_t.max(axis=1),
            traces_at_t.std(axis=1), (traces_at_t**2).mean(axis=1),
            np.abs(np.diff(np.sign(traces_at_t))).sum(axis=1).astype(float)
        ])
        feats.append(feat)

    feats_scaled = scaler.transform(np.array(feats))
    winners      = np.array([som.winner(f) for f in feats_scaled])
    facies_1d    = winners[:, 0] * 4 + winners[:, 1]
    facies_map   = np.tile(facies_1d.reshape(-1, 1), (1, 70))

    facies_vel = {}
    for d in range(70):
        f = facies_1d[d]
        if f not in facies_vel:
            facies_vel[f] = []
        facies_vel[f].append(vmap[d, :].mean())
    facies_vel = {f: np.mean(v) for f, v in facies_vel.items()}

    litho_map = np.zeros((70, 70), dtype=int)
    for f, vel in facies_vel.items():
        clase, _, _ = vel_to_litho(vel)
        litho_map[facies_map == f] = clase

    return litho_map, facies_1d, vmap


def detect_gradient_from_som(litho_map, vmap):
    """
    For FlatVel_A: no faults. Analyzes vertical velocity gradient and layer count.
    Returns: n_layers, gradient (m/s per depth index), vel_min, vel_max, vel_std, homogeneous.
    """
    vel_profile = vmap.mean(axis=1)
    gradient    = np.diff(vel_profile).mean()
    litho_col   = litho_map[:, 35]
    transitions = np.sum(np.diff(litho_col) != 0)

    return {
        'n_layers':    int(transitions + 1),
        'gradient':    round(float(gradient), 2),
        'vel_min':     round(float(vel_profile.min()), 1),
        'vel_max':     round(float(vel_profile.max()), 1),
        'vel_std':     round(float(vel_profile.std()), 1),
        'homogeneous': vel_profile.std() < 200
    }


def analyze_seismic_vision_vel(fig, openai_key, litho_map, vmap, max_width=1200):
    """
    Vision LLM call adapted for FlatVel_A.
    Uses gradient analysis context instead of fault detection.
    """
    buf_raw = io.BytesIO()
    fig.savefig(buf_raw, format='png', dpi=120, bbox_inches='tight')
    buf_raw.seek(0)

    img_pil = Image.open(buf_raw)
    w, h = img_pil.size
    if w > max_width:
        img_pil = img_pil.resize((max_width, int(h * max_width / w)), Image.LANCZOS)

    buf = io.BytesIO()
    img_pil.convert('RGB').save(buf, format='JPEG', quality=82)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)

    grad_info = detect_gradient_from_som(litho_map, vmap)
    som_context = (
        f"CONTEXT FROM SOM ANALYSIS: {grad_info['n_layers']} lithological layer(s) detected. "
        f"Mean velocity gradient = {grad_info['gradient']:.1f} m/s per depth index "
        f"({'increasing' if grad_info['gradient'] > 0 else 'decreasing'} with depth). "
        f"Velocity range: {grad_info['vel_min']:.0f}–{grad_info['vel_max']:.0f} m/s. "
        f"{'Nearly homogeneous sample.' if grad_info['homogeneous'] else 'Significant velocity variation.'}"
    )

    prompt = f"""{som_context}

You are an expert geophysicist. Analyze this seismic image.
THREE panels: LEFT=seismic waveform, CENTER=velocity map (m/s), RIGHT=SOM lithology.

IMPORTANT: This is OpenFWI FlatVel_A — flat horizontal layers, NO faults.
Focus on sedimentary layering and compaction trends.
Axes show pixel indices (0–70), NOT metric units.

1. LAYERS: How many distinct horizontal layers in CENTER? Depth index ranges.
2. VELOCITY TREND: Increases, decreases, or constant with depth?
3. LITHOLOGY: What rock types in RIGHT panel? Top vs bottom.
4. SEISMIC PATTERN: Reflection patterns in LEFT panel.
5. GEOLOGICAL SCENARIO: What depositional environment fits this velocity profile?"""

    r = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {openai_key}", "Content-Type": "application/json"},
        json={"model": "gpt-4o-mini", "max_tokens": 500,
              "messages": [{"role": "user", "content": [
                  {"type": "image_url",
                   "image_url": {"url": f"data:image/jpeg;base64,{img_b64}", "detail": "high"}},
                  {"type": "text", "text": prompt}
              ]}]},
        timeout=60
    )

    if r.status_code != 200:
        return {"error": r.text[:200]}

    resp  = r.json()
    usage = resp['usage']
    cost  = (usage['prompt_tokens'] * 0.00015 + usage['completion_tokens'] * 0.0006) / 1000

    return {
        "text":          resp['choices'][0]['message']['content'],
        "prompt_tokens": usage['prompt_tokens'],
        "cost_usd":      round(cost, 5),
        "grad_info":     grad_info
    }

print("✅ FlatVel_A pipeline functions ready")

## 9. Full Pipeline — FlatVel_A (3 samples)

Selected by velocity variability extremes (computed in Section 2.3):
- **idx=194**: high variability → many distinct layers, active compaction
- **idx=69**: median → typical layered sedimentary section  
- **idx=347**: low variability → nearly homogeneous, shallow burial

**Validation criteria:** Reports should describe horizontal layering and velocity increase 
with depth — and should NOT mention faults.

In [ ]:
samples_vel = [
    (194, 'FlatVel_A — high variability (std≈1219 m/s)'),
    (69,  'FlatVel_A — median variability (std≈648 m/s)'),
    (347, 'FlatVel_A — low variability (std≈92 m/s)'),
]

all_results_vel = {}

for idx, label in samples_vel:
    print(f"\n{'='*60}")
    print(f"📍 {label} (idx={idx})")
    print('='*60)

    # Layer 1 — SOM
    litho_map, facies_1d, vmap = compute_litho_map_vel(idx, data_v, model_v, som2, scaler2)
    fig = make_seismic_plot(idx, data_v, model_v, litho_map)

    # Layer 2 — Vision LLM (gradient context)
    print("🔭 Vision LLM...")
    vision = analyze_seismic_vision_vel(fig, OPENAI_KEY, litho_map, vmap)
    grad   = vision['grad_info']
    print(f"   ${vision['cost_usd']} | {vision['prompt_tokens']} tokens")
    print(f"   SOM: {grad['n_layers']} layers | gradient={grad['gradient']} m/s/idx | "
          f"vel {grad['vel_min']:.0f}–{grad['vel_max']:.0f} m/s")

    # Layer 3 — RAG
    print("📚 RAG arXiv...")
    rag = rag_enrich_interpretation(vision['text'], OPENAI_KEY, verbose=False)
    print(f"   ${rag['cost_usd']} | {len(rag['papers'])} papers")

    total = vision['cost_usd'] + rag['cost_usd']
    all_results_vel[idx] = {
        'label': label, 'vision': vision['text'], 'report': rag['report'],
        'papers': rag['papers'], 'grad_info': grad, 'cost': total
    }

    print(f"\n{rag['report']}")
    print("\n📎 References:")
    for i, p in enumerate(rag['papers']):
        print(f"   [{i+1}] {p['title'][:65]}...")
    print(f"\n💰 Sample cost: ${total:.5f}")

print(f"\n{'='*60}")
total_vel   = sum(r['cost'] for r in all_results_vel.values())
total_fault = sum(r['cost'] for r in all_results.values())
grand_total = total_vel + total_fault
print(f"FlatVel_A TOTAL  : ${total_vel:.5f}")
print(f"FlatFault_A TOTAL: ${total_fault:.5f}")
print(f"GRAND TOTAL      : ${grand_total:.5f}")
print(f"Estimated remaining (from $5.00): ~${5.00 - grand_total:.4f}")

---
## 10. Summary and Future Work

### Results

| Dataset | Samples | SOM QE | LLM fault detection | Total cost |
|---------|---------|--------|---------------------|------------|
| FlatVel_A | 3 | 0.88 | N/A (no faults) | ~$0.011 |
| FlatFault_A | 3 | 0.44 | ✅ Correct (with SOM context) | ~$0.010 |
| **Total** | **6** | — | — | **~$0.021** |

### Key findings

1. **SOM bridge layer is critical** — injecting SOM fault scores as context reduced false positives in GPT-4o mini from 3/3 to 1/3 for no-fault samples.
2. **Image optimization matters** — resizing to 1200px JPEG cut token usage 2.4× vs. raw PNG with no quality loss for geological interpretation.
3. **arXiv RAG provides scientifically grounded context** — papers on FWI with CNNs and multi-scale inversion are directly relevant and consistently retrieved.
4. **Cost is negligible** — ~$0.003/sample makes large-scale batch analysis viable.

### Known limitations

- GPT-4o mini sometimes misidentifies fault location (lateral index can be off by ±10–15 indices)
- The SOM fault threshold (0.3) is empirical — needs calibration on more samples
- `VEL_RANGES` label names leaked into RAG reports in Spanish (now fixed to English)

### Future work

- **CurveVel_A**: curved velocity layers require the 2D pixel-wise SOM (like FlatFault_A), not the 1D row-wise approach used for FlatVel_A. The pipeline structure is ready — only `compute_litho_map` and the prompt context need adaptation.
- **Layer 3 v2**: replace arXiv live search with FAISS local index over pre-downloaded FWI papers for faster, more controlled retrieval.
- **Quantitative evaluation**: compare SOM lithology maps against ground truth velocity bins using IoU or accuracy metrics.
- **GitHub + LinkedIn writeup**: share as open-source pipeline with reproducible Kaggle notebook.